In [ ]:
import pandas as pd

In [ ]:


raw_df = pd.read_parquet('~/Downloads/goiener_post_df.parquet')
raw_df.info()

In [ ]:

grouped_by_customer_id_df = raw_df.groupby('customer_id').agg(
    first_timestamp=('timestamp', 'min'),
    last_timestamp=('timestamp', 'max')
)


start = grouped_by_customer_id_df['first_timestamp'].max()
end = grouped_by_customer_id_df['last_timestamp'].min()

print(f"Data is available from {start} to {end}")





In [ ]:
# Si start no es la hora 0:00 del dia, tomo la hora 0:00 del dia siguiente
if start.hour != 0 or start.minute != 0 or start.second != 0:
    start = (start + pd.Timedelta(days=1)).replace(hour=0, minute=0, second=0)

# Si end no es la hora 23:00 del dia, tomo la hora 23:00 del dia anterior
if end.hour != 23 or end.minute != 0 or end.second != 0:
    end = (end - pd.Timedelta(days=1)).replace(hour=23, minute=0, second=0)

In [ ]:
raw_df = raw_df[(raw_df['timestamp'] >= start) & (raw_df['timestamp'] <= end)]

In [ ]:
raw_df.to_parquet('goiener_same_range.parquet')

In [ ]:
raw_df = pd.read_parquet('goiener_same_range.parquet')

In [ ]:
raw_df.head()

In [ ]:
cluster_input_df = raw_df.pivot(index='customer_id', columns='timestamp', values='kWh')

In [ ]:
len(cluster_input_df)

In [ ]:
cluster_input_df.head()

In [ ]:
cluster_output_df = cluster_input_df

In [ ]:
from sklearn.cluster import KMeans

n_clusters = 5

kmeans = KMeans(n_clusters=n_clusters, random_state=42)

kmeans_result = kmeans.fit_transform(cluster_input_df)

cluster_output_df['cluster'] = kmeans.labels_

kmeans_result

In [ ]:
cluster_output_df.reset_index(inplace=True)

In [ ]:
mins = kmeans_result.min(axis=1)
mins.max()

In [ ]:
mins.min()

In [ ]:
raw_df.head()

In [ ]:
clustered_df = raw_df.drop(columns=['imputed']).groupby(['cluster','timestamp'], as_index=False).mean(numeric_only=True)
clustered_df

In [ ]:
len(raw_df)

In [ ]:
cluster_output_df.groupby('cluster')['customer_id'].nunique()

In [ ]:
metada = pd.read_csv('~/Downloads/metadata.csv')
metada.info()

In [ ]:
metada = metada.dropna(subset=['contracted_tariff'])
metada['contracted_tariff'].unique()

In [ ]:
raw_df.head()

In [ ]:
house_hold_df = raw_df
house_hold_df['user'] = house_hold_df['customer_id']
metada = metada[metada['contracted_tariff'].str.startswith('2')]
house_hold_df = pd.merge(house_hold_df, metada, on='user', how='left')

In [ ]:
house_hold_df.info()

In [ ]:
house_hold_df.groupby('cluster')['user'].nunique()